<a href="https://colab.research.google.com/github/arkeos-kzenon/online-archive/blob/main/upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install spandrel -q
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -P /content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 9.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
name = list(files.upload())[0]

Saving Golden Patintero in a Filipino Village.png to Golden Patintero in a Filipino Village (1).png


In [ ]:
import torch, numpy as np
from PIL import Image
from spandrel import ModelLoader

dev = 'cuda'
model = ModelLoader().load_from_file('/content/RealESRGAN_x4plus_anime_6B.pth').to(dev).eval().half()
s = model.scale

def upscale(img, tile=256, pad=16):
    x = torch.from_numpy(np.array(img.convert('RGB'))).permute(2, 0, 1).float().div(255).unsqueeze(0)
    h, w = x.shape[2], x.shape[3]
    out = torch.zeros(1, 3, h * s, w * s)
    for y0 in range(0, h, tile):
        for x0 in range(0, w, tile):
            y1, x1 = min(y0 + tile, h), min(x0 + tile, w)
            ya, xa = max(y0 - pad, 0), max(x0 - pad, 0)
            yb, xb = min(y1 + pad, h), min(x1 + pad, w)
            t = x[:, :, ya:yb, xa:xb].to(dev).half()
            with torch.no_grad():
                o = model(t).float().cpu()
            oy, ox = (y0 - ya) * s, (x0 - xa) * s
            out[:, :, y0 * s:y1 * s, x0 * s:x1 * s] = o[:, :, oy:oy + (y1 - y0) * s, ox:ox + (x1 - x0) * s]
    arr = out.squeeze(0).clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).numpy()
    return Image.fromarray(arr)

res = upscale(Image.open('/content/' + name)).resize((2560, 1440), Image.LANCZOS)
res.save('/content/kalye_background.webp', quality=92)
files.download('/content/kalye_background.webp')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>